# 03. RAG Pipeline — ChromaDB + LangGraph RAG Agent

## 학습 목표
1. ChromaDB 벡터 스토어 구축 (문서 ingestion → chunking → embedding → 저장)
2. Retrieval 파이프라인 구현 (쿼리 → 유사도 검색 → 컨텍스트 조립)
3. RAG Agent 구현 (LangGraph 기반 — 질문 → 검색 → Claude 답변)
4. Notebook 02의 Vision Agent와 연결 준비 (다음 단계: Notebook 04)

## 전체 흐름
```
문서 파일들
    ↓  (TextLoader / 직접 생성)
Document 객체
    ↓  (RecursiveCharacterTextSplitter)
Chunks (청크)
    ↓  (OllamaEmbeddings)
Vector Embeddings
    ↓  (ChromaDB)
벡터 스토어 (디스크 저장)
    ↑  (similarity_search)
RAG Retriever Agent (LangGraph)
    ↓
Claude API (컨텍스트 + 질문)
    ↓
구조화된 답변
```

In [ ]:
# ── 한글 폰트 & 환경 설정
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import warnings
warnings.filterwarnings('ignore')

print('환경 설정 완료')

In [ ]:
# 패키지 설치 (최초 1회)
# !pip install chromadb langchain langchain-community langchain-text-splitters
# !pip install langchain-ollama anthropic python-dotenv
print('패키지 설치 완료 (이미 설치된 경우 주석 유지)')

In [ ]:
# ── 패키지 버전 확인
import chromadb
import langchain
import anthropic

print(f'chromadb  : {chromadb.__version__}')
print(f'langchain : {langchain.__version__}')
print(f'anthropic : {anthropic.__version__}')

---
## Part 1. 지식 베이스 문서 준비

RAG의 핵심은 **검색 대상 문서**입니다.  
여기서는 컴퓨터 비전 도메인 지식을 샘플 문서로 사용합니다.

실제 프로젝트에서는:
- 논문 PDF → `PyPDFLoader`
- 웹 문서 → `WebBaseLoader`
- 텍스트 파일 → `TextLoader`

In [ ]:
# ── 샘플 지식 베이스 문서 생성
# (실제 파일이 없을 경우를 대비한 인라인 샘플)

import os
from pathlib import Path

KB_DIR = Path('../data/knowledge_base')
KB_DIR.mkdir(parents=True, exist_ok=True)

# 샘플 문서 3개 작성
docs_content = {
    'yolo_overview.txt': """YOLO (You Only Look Once) 객체 탐지 알고리즘

YOLO는 단일 신경망이 전체 이미지를 한 번에 처리하여 바운딩 박스와 클래스 확률을 직접 예측하는 실시간 객체 탐지 시스템입니다.

핵심 아이디어:
- 이미지를 S×S 그리드로 분할
- 각 그리드 셀이 B개의 바운딩 박스와 신뢰도 점수를 예측
- 각 셀이 C개의 클래스 조건부 확률을 예측
- 단일 forward pass로 탐지 완료 → 실시간 처리 가능

YOLOv8 특징:
- Anchor-free 탐지 헤드
- C2f(Cross Stage Partial with 2 convolutions) 블록
- mAP50-95 기준 COCO 데이터셋에서 최고 수준 성능
- 탐지(Detection), 분류(Classification), 세그멘테이션(Segmentation), 포즈(Pose) 지원
- 모델 크기: nano(n), small(s), medium(m), large(l), extra-large(x)

활용 사례:
- 자율주행: 차량, 보행자, 신호등 탐지
- 보안: CCTV 이상 감지
- 의료: X-ray 이상 부위 탐지
- 제조: 불량품 검사

사용법 예시 (Python):
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
results = model('image.jpg')
for r in results:
    boxes = r.boxes  # 바운딩 박스
    masks = r.masks  # 세그멘테이션 마스크
""",
    'sam_overview.txt': """SAM (Segment Anything Model) — Meta AI

SAM은 Meta AI가 개발한 범용 이미지 세그멘테이션 모델로, 어떤 객체도 프롬프트 한 번으로 분할할 수 있습니다.

핵심 구조:
1. Image Encoder: MAE(Masked Autoencoder) 사전학습된 ViT-H
   - 이미지를 64×64 임베딩으로 변환
2. Prompt Encoder: 포인트, 박스, 텍스트, 마스크 입력 처리
   - Sparse prompt (포인트, 박스): positional encoding
   - Dense prompt (마스크): convolution으로 downsampling
3. Mask Decoder: Transformer 기반 디코더
   - IoU 점수와 함께 3개의 마스크 후보 출력

프롬프트 종류:
- 포인트 프롬프트: 객체 위의 점 하나 클릭
- 박스 프롬프트: 관심 영역(RoI) 바운딩 박스
- 자동 모드: 이미지 전체에 그리드 포인트 생성 → 모든 객체 자동 탐지

SAM2 (2024) 개선사항:
- 비디오 세그멘테이션 지원
- Streaming memory 구조로 시간축 추적 가능
- 속도 6배 향상

활용 사례:
- 의료 이미징: 종양, 장기 세그멘테이션
- 자율주행: 도로, 차선, 장애물 분할
- AR/VR: 실시간 객체 분리
- 이커머스: 상품 배경 제거
""",
    'depth_estimation.txt': """DepthAnything — 단안 깊이 추정 모델

DepthAnything(2024)은 단일 RGB 이미지에서 픽셀별 깊이값을 추정하는 Foundation Model입니다.

핵심 특징:
- 62M개의 라벨 없는 이미지로 사전학습 (DINOv2 backbone)
- 라벨 있는 데이터 147K장으로 지식 증류(Knowledge Distillation)
- 상대적 깊이(Relative Depth)와 절대적 깊이(Metric Depth) 모두 지원
- Indoor/Outdoor 구분 없이 범용적으로 작동

모델 구조:
- Encoder: DINOv2 ViT-L (대형 Vision Transformer)
- Decoder: DPT (Dense Prediction Transformer)
- 출력: 256×256 깊이 맵 (실수값, 0=가까움, 1=멂)

DepthAnything v2 (2024.06) 개선:
- 합성 데이터(Synthetic Data) 추가 학습
- 세밀한 경계 추정 향상
- 실내 환경 정확도 크게 향상

활용 사례:
- 자율주행: 3D 장면 이해, 충돌 회피
- 로보틱스: 물체 파지(grasping) 계획
- AR: 실제 환경에 가상 객체 자연스럽게 합성
- 사진 편집: 배경 흐림 처리(bokeh effect)

사용법 예시:
from transformers import pipeline
pipe = pipeline('depth-estimation', model='depth-anything/Depth-Anything-V2-Small-hf')
depth = pipe('image.jpg')['depth']
""",
    'langgraph_concepts.txt': """LangGraph — 에이전트 상태머신 프레임워크

LangGraph는 LangChain 위에서 동작하는 에이전트 오케스트레이션 프레임워크입니다.
에이전트를 유향 그래프(Directed Graph)로 정의하여 복잡한 워크플로우를 구현합니다.

핵심 개념:

1. StateGraph
   - 에이전트의 전체 상태를 담는 TypedDict
   - 모든 노드가 같은 State를 공유
   - Annotated[List, add_messages] 패턴으로 메시지 누적

2. Node (노드)
   - 실제 처리 로직: LLM 호출, 툴 실행, 조건 판단
   - 입력: State → 출력: State 업데이트 dict
   - graph.add_node(name, function)으로 등록

3. Edge (엣지)
   - 노드 간 연결
   - 일반 엣지: add_edge(A, B)
   - 조건부 엣지: add_conditional_edges(A, condition_fn, {result: next_node})

4. Checkpointer (체크포인터)
   - 각 스텝의 State를 저장
   - Human-in-the-loop: interrupt_before/after 설정
   - MemorySaver: 인메모리 저장

5. ToolNode
   - 툴 호출을 자동으로 처리하는 특수 노드
   - LLM이 tool_call을 반환하면 실행 → 결과를 메시지에 추가

멀티 에이전트 패턴:
- Supervisor: 여러 에이전트를 조율하는 상위 에이전트
- Handoff: 특정 에이전트로 작업 위임
- Parallel: 여러 에이전트 동시 실행 후 결과 합산
""",
    'rag_concepts.txt': """RAG (Retrieval-Augmented Generation) 개념

RAG는 LLM이 학습 데이터에 없는 정보를 외부 지식 베이스에서 검색하여 답변하는 패턴입니다.

구성 요소:

1. 문서 처리 (Indexing)
   - 문서 로드: PDF, TXT, HTML 등 다양한 형식
   - 청킹(Chunking): 문서를 적절한 크기로 분할
     * chunk_size: 청크당 문자 수 (보통 500~1000)
     * chunk_overlap: 청크 간 겹치는 문자 수 (컨텍스트 유지)
   - 임베딩: 텍스트 → 고차원 벡터
   - 벡터 스토어: ChromaDB, Pinecone, FAISS 등에 저장

2. 검색 (Retrieval)
   - 쿼리 임베딩: 질문도 동일한 임베딩 모델로 변환
   - 유사도 검색: 코사인 유사도로 가장 가까운 k개 청크 반환
   - MMR (Maximal Marginal Relevance): 다양성을 고려한 검색

3. 생성 (Generation)
   - 검색된 청크를 컨텍스트로 LLM에 전달
   - 프롬프트: "다음 컨텍스트를 바탕으로 질문에 답하시오: {context}\n질문: {query}"
   - LLM이 컨텍스트 기반 답변 생성

RAG의 장점:
- 최신 정보 반영 (LLM 학습 이후 데이터)
- 할루시네이션(Hallucination) 감소
- 출처 추적 가능
- 도메인 특화 지식 적용 용이

Advanced RAG 기법:
- HyDE (Hypothetical Document Embedding): 가상 답변 먼저 생성 → 임베딩
- Query Expansion: 쿼리를 여러 버전으로 확장
- Reranking: Cross-encoder로 검색 결과 재정렬
"""
}

# 파일로 저장
for filename, content in docs_content.items():
    (KB_DIR / filename).write_text(content, encoding='utf-8')

print(f'지식 베이스 문서 {len(docs_content)}개 생성 완료: {KB_DIR.resolve()}')
for f in KB_DIR.iterdir():
    print(f'  - {f.name} ({f.stat().st_size} bytes)')

---
## Part 2. 문서 로드 & 청킹 (Chunking)

**왜 청킹이 필요한가?**
- LLM의 컨텍스트 윈도우에 문서 전체를 넣을 수 없음
- 긴 문서를 검색하면 관련 없는 내용이 섞임
- 적절한 크기로 분할해야 검색 정확도가 올라감

**RecursiveCharacterTextSplitter**  
우선순위: `\n\n` → `\n` → ` ` → 문자 단위로 분할  
문단 → 줄 → 단어 순서로 자연스러운 분할

In [ ]:
# ── 문서 로드
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# DirectoryLoader로 KB_DIR의 모든 .txt 파일 로드
loader = DirectoryLoader(
    str(KB_DIR),
    glob='*.txt',
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'}
)
raw_docs = loader.load()

print(f'로드된 문서 수: {len(raw_docs)}')
for doc in raw_docs:
    print(f'  - {doc.metadata["source"].split("/")[-1].split(chr(92))[-1]} | {len(doc.page_content)}자')

In [ ]:
# ── 청킹
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,       # 청크당 최대 문자 수
    chunk_overlap=80,     # 청크 간 겹치는 문자 (컨텍스트 유지)
    length_function=len,
    separators=['\n\n', '\n', ' ', '']  # 분할 우선순위
)

chunks = splitter.split_documents(raw_docs)

print(f'원본 문서: {len(raw_docs)}개 → 청크: {len(chunks)}개')
print(f'\n청크 크기 분포:')
sizes = [len(c.page_content) for c in chunks]
print(f'  최소: {min(sizes)}자 | 평균: {sum(sizes)//len(sizes)}자 | 최대: {max(sizes)}자')

# 샘플 청크 확인
print(f'\n샘플 청크 (첫 번째):')
print(f'  출처: {chunks[0].metadata["source"].split(chr(92))[-1].split("/")[-1]}')
print(f'  내용: {chunks[0].page_content[:200]}...')

---
## Part 3. 임베딩 & ChromaDB 벡터 스토어 구축

**임베딩 모델 선택:**  
- `OllamaEmbeddings`: Ollama로 로컬에서 실행 (nomic-embed-text 권장)
- `FakeEmbeddings`: 테스트용 (Ollama 없을 때 사용)

**ChromaDB:**  
- 로컬 디스크에 저장 (`persist_directory`)
- `collection_name`으로 여러 지식 베이스 분리 관리
- 재실행 시 기존 데이터 로드 가능

In [ ]:
# -- 임베딩 모델 설정
import requests

def check_ollama_running() -> bool:
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=2)
        if r.status_code != 200:
            return False
        # nomic-embed-text 모델 존재 여부 확인
        models = [m["name"] for m in r.json().get("models", [])]
        return any("nomic-embed-text" in m for m in models)
    except Exception:
        return False

OLLAMA_RUNNING = check_ollama_running()
print(f'Ollama + nomic-embed-text: {"사용 가능" if OLLAMA_RUNNING else "없음 → FakeEmbeddings 사용"}')

if OLLAMA_RUNNING:
    from langchain_ollama import OllamaEmbeddings
    embeddings = OllamaEmbeddings(model="nomic-embed-text")
    EMBED_DIM = 768
    print("OllamaEmbeddings (nomic-embed-text) 초기화")
else:
    from langchain_core.embeddings.fake import FakeEmbeddings
    embeddings = FakeEmbeddings(size=768)
    EMBED_DIM = 768
    print("FakeEmbeddings (768차원) — 학습용 모드")
    print("tip: 실제 임베딩 원하면 터미널에서 → ollama pull nomic-embed-text")


In [ ]:
# -- ChromaDB 벡터 스토어 구축
from langchain_community.vectorstores import Chroma
import chromadb
from pathlib import Path

CHROMA_DIR = str(Path("../data/chroma_db").resolve())
COLLECTION_NAME = "cv_knowledge_base"

# 디렉토리 생성
Path(CHROMA_DIR).mkdir(parents=True, exist_ok=True)

# 기존 컬렉션만 삭제 (파일 시스템 건드리지 않음)
try:
    _client = chromadb.PersistentClient(path=CHROMA_DIR)
    _client.delete_collection(COLLECTION_NAME)
    print("기존 컬렉션 삭제")
    del _client
except Exception:
    pass  # 없으면 그냥 진행

# 벡터 스토어 생성
print(f"{len(chunks)}개 청크 임베딩 중...")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=CHROMA_DIR
)

print("ChromaDB 구축 완료!")
print(f"  저장 경로: {CHROMA_DIR}")
print(f"  저장된 벡터 수: {vectorstore._collection.count()}")


In [ ]:
# ── 유사도 검색 테스트
# 쿼리 → 임베딩 → ChromaDB 코사인 유사도 검색 → 상위 k개 청크 반환

test_queries = [
    'YOLO 모델의 특징은 무엇인가요?',
    'SAM은 어떻게 세그멘테이션을 수행하나요?',
    'RAG에서 청킹이란 무엇인가요?',
]

for query in test_queries:
    results = vectorstore.similarity_search(query, k=2)
    print(f'\n쿼리: "{query}"')
    for i, doc in enumerate(results, 1):
        src = doc.metadata['source'].split('\\')[-1].split('/')[-1]
        print(f'  [{i}] {src}: {doc.page_content[:100]}...')

In [ ]:
# ── 유사도 점수 포함 검색 (similarity_search_with_score)
# 점수가 낮을수록 유사도가 높음 (L2 거리 기반)

query = 'DepthAnything 깊이 추정 모델 구조'
results_with_score = vectorstore.similarity_search_with_score(query, k=3)

print(f'쿼리: "{query}"')
print(f'(낮은 점수 = 높은 유사도)')
for doc, score in results_with_score:
    src = doc.metadata['source'].split('\\')[-1].split('/')[-1]
    print(f'  점수: {score:.4f} | {src}: {doc.page_content[:80]}...')

---
## Part 4. Retriever 인터페이스 구성

LangChain의 `Retriever` 인터페이스로 감싸면  
LangGraph 노드에서 `.invoke(query)` 한 줄로 검색 가능합니다.

In [ ]:
# ── Retriever 생성
retriever = vectorstore.as_retriever(
    search_type='similarity',  # 'mmr' (다양성 고려) 도 가능
    search_kwargs={'k': 3}     # 상위 3개 청크 반환
)

# 테스트
docs = retriever.invoke('LangGraph 에이전트 상태머신이란?')
print(f'검색된 청크: {len(docs)}개')
for i, doc in enumerate(docs, 1):
    src = doc.metadata['source'].split('\\')[-1].split('/')[-1]
    print(f'\n[{i}] {src}')
    print(f'    {doc.page_content[:150]}...')

In [ ]:
# ── 컨텍스트 조립 유틸리티
# 검색된 청크들을 하나의 컨텍스트 문자열로 합치는 헬퍼 함수

def format_context(docs: list, max_chars: int = 2000) -> str:
    """검색된 Document 목록 → 컨텍스트 문자열"""
    context_parts = []
    total = 0
    for i, doc in enumerate(docs, 1):
        src = doc.metadata['source'].split('\\')[-1].split('/')[-1]
        text = f'[출처 {i}: {src}]\n{doc.page_content}'
        if total + len(text) > max_chars:
            break
        context_parts.append(text)
        total += len(text)
    return '\n\n'.join(context_parts)

# 테스트
sample_docs = retriever.invoke('YOLO 사용법')
context = format_context(sample_docs)
print(f'컨텍스트 길이: {len(context)}자')
print(f'\n--- 컨텍스트 미리보기 ---')
print(context[:400], '...')

---
## Part 5. Claude API 연동 RAG 체인

`retriever → context → Claude API`  
이 흐름을 LangGraph State로 관리합니다.

Claude API 직접 호출 방식을 사용합니다  
(LangChain의 `ChatAnthropic`도 가능하지만 직접 호출이 더 명확)

In [ ]:
# -- LLM 설정 (Ollama — 완전 무료)
from langchain_ollama import ChatOllama
import requests

OLLAMA_MODEL = "llama3.2"  # ollama pull llama3.2

def check_ollama_model(model: str) -> bool:
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=2)
        models = [m["name"] for m in r.json().get("models", [])]
        return any(model.split(":")[0] in m for m in models)
    except:
        return False

USE_OLLAMA = check_ollama_model(OLLAMA_MODEL)
print(f"Ollama ({OLLAMA_MODEL}): {"사용 가능" if USE_OLLAMA else "없음 → ollama pull llama3.2 실행 필요"}")

if USE_OLLAMA:
    llm = ChatOllama(model=OLLAMA_MODEL, temperature=0)
    print("ChatOllama 초기화 완료")


In [ ]:
# -- Ollama RAG 함수 (Claude API 대체)
from langchain_core.messages import SystemMessage, HumanMessage

RAG_SYSTEM_PROMPT = """당신은 컴퓨터 비전과 머신러닝 전문 어시스턴트입니다.
제공된 컨텍스트를 바탕으로 정확하고 간결하게 답변합니다.
컨텍스트에 없는 내용은 솔직하게 모른다고 말합니다.
답변은 한국어로 작성합니다."""

def rag_with_ollama(query: str, context: str) -> str:
    """Ollama를 사용한 RAG 답변 생성"""
    if not USE_OLLAMA:
        return f"[Mock] 쿼리: "{query}" | 컨텍스트 {len(context)}자 기반 답변"
    
    user_message = f"""다음 컨텍스트를 참고하여 질문에 답변해주세요.

## 컨텍스트
{context}

## 질문
{query}"""
    
    messages = [
        SystemMessage(content=RAG_SYSTEM_PROMPT),
        HumanMessage(content=user_message)
    ]
    response = llm.invoke(messages)
    return response.content

# 하위 호환성 유지 (기존 코드에서 rag_with_claude 호출 시 자동 연결)
rag_with_claude = rag_with_ollama

# 테스트
test_query = "YOLOv8의 특징을 3가지만 설명해주세요"
test_docs = retriever.invoke(test_query)
test_context = format_context(test_docs)
answer = rag_with_ollama(test_query, test_context)
print(f"쿼리: {test_query}")
print(f"답변:
{answer}")


---
## Part 6. LangGraph RAG Agent 구현

지금까지 구성한 검색 + 생성 파이프라인을  
**LangGraph StateGraph**로 오케스트레이션합니다.

```
[질문 입력]
    ↓
 query_analysis   ← 쿼리 분석 및 전처리
    ↓
 retrieval        ← ChromaDB 유사도 검색
    ↓
 generation       ← Claude API 답변 생성
    ↓
 [최종 답변]
```

In [ ]:
# ── RAG Agent State 정의
from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Optional
from langchain_core.documents import Document

class RAGState(TypedDict):
    # 입력
    question: str              # 사용자 질문
    # 중간 상태
    refined_query: str         # 전처리된 쿼리
    retrieved_docs: List[Document]  # 검색된 청크
    context: str               # 조립된 컨텍스트
    # 출력
    answer: str                # 최종 답변
    sources: List[str]         # 참고 출처 목록
    # 메타
    error: Optional[str]       # 에러 메시지 (있을 경우)

print('RAGState 정의 완료')
print('필드:', list(RAGState.__annotations__.keys()))

In [ ]:
# ── 노드 1: 쿼리 분석
# 질문을 검색에 최적화된 형태로 정제

def query_analysis_node(state: RAGState) -> dict:
    """쿼리 전처리 노드
    
    - 불필요한 조사/어미 제거
    - 핵심 키워드 추출
    - (심화) 쿼리 확장: 여러 버전의 쿼리로 변환
    """
    question = state['question']
    
    # 간단한 전처리: 질문 어미 제거
    # 실제로는 LLM으로 더 정교하게 처리 가능
    stop_words = ['무엇인가요', '어떻게', '알려주세요', '설명해주세요', '?', '입니까', '인가요']
    refined = question
    for sw in stop_words:
        refined = refined.replace(sw, '').strip()
    
    print(f'  [query_analysis] "{question}" → "{refined}"')
    return {'refined_query': refined}

print('query_analysis_node 정의 완료')

In [ ]:
# ── 노드 2: 검색 (Retrieval)

def retrieval_node(state: RAGState) -> dict:
    """ChromaDB 유사도 검색 노드"""
    query = state.get('refined_query', state['question'])
    
    # 검색 실행
    docs = retriever.invoke(query)
    
    # 컨텍스트 조립
    context = format_context(docs, max_chars=2500)
    
    # 출처 목록 추출
    sources = list({
        doc.metadata['source'].split('\\')[-1].split('/')[-1]
        for doc in docs
    })
    
    print(f'  [retrieval] {len(docs)}개 청크 검색 | 출처: {sources}')
    return {
        'retrieved_docs': docs,
        'context': context,
        'sources': sources
    }

print('retrieval_node 정의 완료')

In [ ]:
# ── 노드 3: 생성 (Generation)

def generation_node(state: RAGState) -> dict:
    """Claude API 답변 생성 노드"""
    question = state['question']
    context = state.get('context', '')
    
    if not context:
        return {'answer': '관련 정보를 찾을 수 없습니다.', 'error': 'no_context'}
    
    answer = rag_with_claude(question, context)
    
    print(f'  [generation] 답변 생성 완료 ({len(answer)}자)')
    return {'answer': answer, 'error': None}

print('generation_node 정의 완료')

In [ ]:
# ── LangGraph RAG Agent 그래프 구성

def build_rag_graph() -> StateGraph:
    """RAG Agent 그래프 빌드"""
    graph = StateGraph(RAGState)
    
    # 노드 등록
    graph.add_node('query_analysis', query_analysis_node)
    graph.add_node('retrieval', retrieval_node)
    graph.add_node('generation', generation_node)
    
    # 엣지 연결 (선형 파이프라인)
    graph.set_entry_point('query_analysis')
    graph.add_edge('query_analysis', 'retrieval')
    graph.add_edge('retrieval', 'generation')
    graph.add_edge('generation', END)
    
    return graph.compile()

rag_agent = build_rag_graph()
print('RAG Agent 그래프 빌드 완료')

# 그래프 시각화
try:
    from IPython.display import Image, display
    display(Image(rag_agent.get_graph().draw_mermaid_png()))
except Exception as e:
    # 시각화 라이브러리 없을 경우 Mermaid 텍스트 출력
    print('그래프 구조 (Mermaid):')
    print(rag_agent.get_graph().draw_mermaid())

In [ ]:
# ── RAG Agent 실행 테스트

def run_rag_agent(question: str) -> dict:
    """RAG Agent 실행 래퍼"""
    print(f'\n{'='*60}')
    print(f'질문: {question}')
    print('='*60)
    
    initial_state: RAGState = {
        'question': question,
        'refined_query': '',
        'retrieved_docs': [],
        'context': '',
        'answer': '',
        'sources': [],
        'error': None
    }
    
    result = rag_agent.invoke(initial_state)
    
    print(f'\n답변:')
    print(result['answer'])
    print(f'\n참고 출처: {", ".join(result["sources"])}')
    return result

# 테스트 1
r1 = run_rag_agent('SAM 모델의 프롬프트 종류를 알려주세요')

In [ ]:
# 테스트 2
r2 = run_rag_agent('LangGraph에서 조건부 엣지는 어떻게 사용하나요?')

In [ ]:
# 테스트 3 — 지식 베이스에 없는 질문
r3 = run_rag_agent('GPT-4의 아키텍처를 설명해주세요')

---
## Part 7. 조건부 RAG — 검색 품질 평가 후 재검색

검색 결과가 불충분하면 쿼리를 재작성하여 다시 검색하는  
**Self-Corrective RAG** 패턴을 구현합니다.

```
[질문]
  → query_analysis
  → retrieval
  → relevance_check  ← 검색 결과 관련성 평가
      ↓ 충분         ↓ 부족
  generation      query_rewrite  → retrieval (재검색)
      ↓
  [답변]
```

In [ ]:
# ── 확장 State (재시도 카운터 추가)
class AdvancedRAGState(TypedDict):
    question: str
    refined_query: str
    retrieved_docs: List[Document]
    context: str
    answer: str
    sources: List[str]
    error: Optional[str]
    # 추가 필드
    relevance_score: float    # 검색 결과 관련성 점수 (0~1)
    retry_count: int          # 재시도 횟수
    rewrite_history: List[str]  # 쿼리 재작성 이력

print('AdvancedRAGState 정의 완료')

In [ ]:
# ── 관련성 평가 노드

def relevance_check_node(state: AdvancedRAGState) -> dict:
    """검색된 청크의 관련성을 평가하는 노드
    
    간단한 키워드 기반 평가 (실제로는 Cross-encoder 모델 사용)
    """
    question = state['question'].lower()
    docs = state['retrieved_docs']
    
    if not docs:
        return {'relevance_score': 0.0}
    
    # 질문의 핵심 단어가 검색 결과에 얼마나 포함되는지 계산
    # (실제 프로젝트: Cross-encoder 또는 LLM 기반 평가)
    question_words = set(question.replace('?', '').split())
    stop = {'은', '는', '이', '가', '을', '를', '의', '에', '서', '로', '과', '와', '도', '만'}
    question_words -= stop
    
    combined_content = ' '.join(d.page_content.lower() for d in docs)
    matched = sum(1 for w in question_words if w in combined_content)
    score = matched / max(len(question_words), 1)
    
    print(f'  [relevance_check] 점수: {score:.2f} (임계값: 0.3)')
    return {'relevance_score': score}


def relevance_router(state: AdvancedRAGState) -> str:
    """관련성 점수 기반 라우팅"""
    score = state.get('relevance_score', 0.0)
    retry = state.get('retry_count', 0)
    
    if score >= 0.3 or retry >= 2:  # 충분하거나 최대 재시도 도달
        return 'generate'
    return 'rewrite'


def query_rewrite_node(state: AdvancedRAGState) -> dict:
    """쿼리 재작성 노드 — 다른 키워드로 재검색 유도"""
    question = state['question']
    retry = state.get('retry_count', 0)
    history = state.get('rewrite_history', [])
    
    # 간단한 재작성 전략 (실제: LLM 사용)
    rewrites = [
        question.replace('어떻게', '방법').replace('무엇', '개념'),
        ' '.join(question.split()[:3])  # 앞 3단어만 사용
    ]
    new_query = rewrites[retry % len(rewrites)]
    
    print(f'  [query_rewrite] 재작성: "{new_query}" (시도 {retry+1})')
    return {
        'refined_query': new_query,
        'retry_count': retry + 1,
        'rewrite_history': history + [new_query]
    }

print('관련성 평가 / 재작성 노드 정의 완료')

In [ ]:
# ── Advanced RAG Agent 그래프 구성

def build_advanced_rag_graph():
    """Self-Corrective RAG 그래프"""
    
    # AdvancedRAGState 호환 노드 래퍼
    def advanced_retrieval_node(state: AdvancedRAGState) -> dict:
        query = state.get('refined_query') or state['question']
        docs = retriever.invoke(query)
        context = format_context(docs)
        sources = list({d.metadata['source'].split('\\')[-1].split('/')[-1] for d in docs})
        print(f'  [retrieval] 쿼리: "{query}" → {len(docs)}개')
        return {'retrieved_docs': docs, 'context': context, 'sources': sources}
    
    def advanced_generation_node(state: AdvancedRAGState) -> dict:
        answer = rag_with_claude(state['question'], state.get('context', ''))
        print(f'  [generation] 완료 ({len(answer)}자)')
        return {'answer': answer}
    
    graph = StateGraph(AdvancedRAGState)
    
    graph.add_node('query_analysis', query_analysis_node)
    graph.add_node('retrieval', advanced_retrieval_node)
    graph.add_node('relevance_check', relevance_check_node)
    graph.add_node('query_rewrite', query_rewrite_node)
    graph.add_node('generation', advanced_generation_node)
    
    graph.set_entry_point('query_analysis')
    graph.add_edge('query_analysis', 'retrieval')
    graph.add_edge('retrieval', 'relevance_check')
    graph.add_conditional_edges(
        'relevance_check',
        relevance_router,
        {'generate': 'generation', 'rewrite': 'query_rewrite'}
    )
    graph.add_edge('query_rewrite', 'retrieval')  # 재검색 루프
    graph.add_edge('generation', END)
    
    return graph.compile()

advanced_rag_agent = build_advanced_rag_graph()
print('Advanced RAG Agent 빌드 완료')

try:
    from IPython.display import Image, display
    display(Image(advanced_rag_agent.get_graph().draw_mermaid_png()))
except Exception:
    print(advanced_rag_agent.get_graph().draw_mermaid())

In [ ]:
# ── Advanced RAG Agent 테스트

def run_advanced_rag(question: str):
    print(f'\n{"="*60}')
    print(f'질문: {question}')
    print('='*60)
    
    initial: AdvancedRAGState = {
        'question': question,
        'refined_query': '',
        'retrieved_docs': [],
        'context': '',
        'answer': '',
        'sources': [],
        'error': None,
        'relevance_score': 0.0,
        'retry_count': 0,
        'rewrite_history': []
    }
    result = advanced_rag_agent.invoke(initial)
    
    print(f'\n답변:')
    print(result['answer'])
    print(f'관련성 점수: {result.get("relevance_score", 0):.2f}')
    print(f'재시도: {result.get("retry_count", 0)}회')
    print(f'출처: {", ".join(result["sources"])}')
    return result

# 테스트
r_adv = run_advanced_rag('DepthAnything v2 개선사항은 무엇인가요?')

---
## Part 8. 벡터 스토어 영속성 확인 & 재로드

ChromaDB는 디스크에 저장되므로 **재시작 후에도 재구축 없이 로드** 가능합니다.

In [ ]:
# ── 디스크에서 ChromaDB 재로드
# (새로운 Python 세션에서 이미 만들어진 DB를 불러오는 방법)

def load_vectorstore(persist_dir: str, collection_name: str, embed_fn):
    """기존 ChromaDB 로드"""
    vs = Chroma(
        collection_name=collection_name,
        embedding_function=embed_fn,
        persist_directory=persist_dir
    )
    count = vs._collection.count()
    print(f'ChromaDB 재로드 완료 — 벡터 수: {count}')
    return vs

reloaded_vs = load_vectorstore(CHROMA_DIR, COLLECTION_NAME, embeddings)

# 재로드된 스토어로 검색 테스트
test = reloaded_vs.similarity_search('RAG 검색 방법', k=2)
print(f'재로드 후 검색: {len(test)}개 결과')
for doc in test:
    src = doc.metadata['source'].split('\\')[-1].split('/')[-1]
    print(f'  - {src}: {doc.page_content[:80]}...')

In [ ]:
# ── 문서 추가 (incremental update)
# 새 문서를 기존 DB에 추가하는 방법

new_doc_content = """ReAct (Reasoning and Acting) — 에이전트 추론 패턴

ReAct는 LLM이 추론(Reasoning)과 행동(Acting)을 번갈아 수행하는 에이전트 패턴입니다.

동작 원리:
1. Thought (생각): LLM이 현재 상황을 분석하고 다음 행동을 결정
2. Action (행동): 툴 호출 (검색, 계산, API 등)
3. Observation (관찰): 툴 실행 결과 확인
4. 반복: 목표 달성까지 위 과정 반복

LangGraph에서의 구현:
- Thought → LLM 노드
- Action → ToolNode
- Observation → State 업데이트
- 조건부 엣지로 반복 vs 종료 결정
"""

new_chunks = splitter.create_documents(
    [new_doc_content],
    metadatas=[{'source': 'react_pattern.txt'}]
)

reloaded_vs.add_documents(new_chunks)
print(f'문서 추가 후 벡터 수: {reloaded_vs._collection.count()}')

# 추가된 문서 검색 테스트
result = reloaded_vs.similarity_search('ReAct 패턴 Thought Action', k=2)
for doc in result:
    src = doc.metadata['source'].split('\\')[-1].split('/')[-1]
    print(f'  - {src}: {doc.page_content[:100]}...')

---
## 정리

| 구성 요소 | 역할 | 핵심 API |
|----------|------|----------|
| `DirectoryLoader` | 파일 로드 | `loader.load()` |
| `RecursiveCharacterTextSplitter` | 청킹 | `splitter.split_documents(docs)` |
| `OllamaEmbeddings` | 벡터 변환 | `embeddings.embed_query(text)` |
| `ChromaDB` | 벡터 저장/검색 | `Chroma.from_documents()`, `.similarity_search()` |
| `Retriever` | 검색 인터페이스 | `retriever.invoke(query)` |
| `Claude API` | RAG 답변 생성 | `client.messages.create()` |
| `RAG Agent` | LangGraph 오케스트레이션 | `StateGraph` + 3 노드 |
| `Advanced RAG` | Self-Corrective | 조건부 엣지 + 재검색 루프 |

## 다음 단계: Notebook 04

```
Notebook 02의 Vision Agent
         +
Notebook 03의 RAG Agent
         ↓
Orchestrator → Vision Analyst → RAG Retriever → Report Writer
```

4개의 에이전트가 협력하는 **전체 파이프라인**을 구현합니다.